In [1]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [2]:
db_path = Path("../../data/database.sqlite")
conn = sqlite3.connect(db_path.as_posix())
df_agg = pd.read_sql_query("SELECT * FROM Match_Aggregated", conn)
df_match = pd.read_sql_query("SELECT * FROM Match_Cleaned", conn)
conn.close()

In [3]:
df_match_fc_bayern_munich = df_agg[
    (df_agg["match_api_id"] == 2002277) & (df_agg["team_side"] == "away")
]
df_match_fc_bayern_munich.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name,player_x,player_y
406790,2002277,2016-02-14 00:00:00,away_player_7,30834,away,7,16461,9014,89.0,89.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,2.0,8.0
406794,2002277,2016-02-14 00:00:00,away_player_2,30894,away,2,143229,121939,87.0,87.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,2.0,3.0
406890,2002277,2016-02-14 00:00:00,away_player_6,49939,away,6,17411,181872,85.0,85.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,5.0,6.0
406927,2002277,2016-02-14 00:00:00,away_player_11,93447,away,11,150542,188545,88.0,89.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,5.0,11.0
406961,2002277,2016-02-14 00:00:00,away_player_8,116772,away,8,170800,189596,86.0,88.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,4.0,8.0


In [4]:
df_match_real_madrid = df_agg[
    (df_agg["match_api_id"] == 2030529) & (df_agg["team_side"] == "away")
]
df_match_real_madrid.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name,player_x,player_y
425882,2030529,2016-05-14 00:00:00,away_player_3,25921,away,3,142272,120533,84.0,84.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,4.0,3.0
425887,2030529,2016-05-14 00:00:00,away_player_10,26166,away,10,93534,165153,86.0,86.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,5.0,10.0
425907,2030529,2016-05-14 00:00:00,away_player_5,28467,away,5,110682,176676,83.0,83.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,8.0,3.0
425916,2030529,2016-05-14 00:00:00,away_player_11,30893,away,11,33331,20801,93.0,93.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,7.0,10.0
425925,2030529,2016-05-14 00:00:00,away_player_4,30962,away,4,161328,155862,87.0,87.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,6.0,3.0


# Model 1

In [6]:
class FormationTower(nn.Module):
    def __init__(self):
        super().__init__()
        # Phi: (B, 9, 2) -> (B, 9, 64)
        self.phi = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=64),
            nn.LeakyReLU(),
        )

        # Rho: (B, 64 + 2) -> (B, 64)
        # We match in_features to the phi output (64) + position (2)
        self.rho = nn.Sequential(
            nn.Linear(in_features=64 + 2, out_features=128),
            nn.BatchNorm1d(128),  # Must match out_features above
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),  # Final embedding dim: 64
        )

    def forward(self, formation, position):
        x = self.phi(formation)  # (Batch, 9, 64)
        x = torch.sum(x, dim=1)  # (Batch, 64)

        x = torch.concat([x, position], dim=1)  # (Batch, 66)
        return self.rho(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),  # Final embedding dim: 64
        )

    def forward(self, player):
        return self.mlp(player)

In [7]:
class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.formation_tower = FormationTower()
        self.player_tower = PlayerTower()

    def forward(self, formation, position, player):
        formation_embedding = self.formation_tower(formation, position)
        player_embedding = self.player_tower(player)

        f_norm = formation_embedding / (
            torch.linalg.norm(formation_embedding, dim=1, keepdim=True) + 1e-8
        )
        p_norm = player_embedding / (
            torch.linalg.norm(player_embedding, dim=1, keepdim=True) + 1e-8
        )

        similarity_score = (f_norm * p_norm).sum(dim=1)

        return f_norm, p_norm, similarity_score

In [8]:
player_stats_cols = [
    "preferred_foot",
    "attacking_work_rate",
    "defensive_work_rate",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",
    "dribbling",
    "curve",
    "free_kick_accuracy",
    "long_passing",
    "ball_control",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "marking",
    "standing_tackle",
    "sliding_tackle",
]

In [10]:
import plotly.graph_objects as go
import numpy as np
import torch
import pandas as pd


def visualize_tactical_fit(custom_coords, squad_df, model, device, player_stats_cols):
    """
    custom_coords: List of tuples [(x1, y1), (x2, y2), ... (x10, y10)]
    squad_df: DataFrame containing the players you want to test (e.g., FC Bayern)
    """
    model.eval()

    # 1. Prepare Coordinates
    coords_np = np.array(custom_coords, dtype=float)
    # Normalize globally (consistent with training: x centered at 5, y at 7)
    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    all_rankings = []

    # 2. Process each of the 10 positions
    for i in range(len(coords_norm)):
        target_pos_val = coords_norm[i]
        teammates_pos_vals = np.delete(coords_norm, i, axis=0)

        # Calculate Relative Geometry (matching advanced model logic)
        relative_vectors = teammates_pos_vals - target_pos_val
        distances = np.linalg.norm(relative_vectors, axis=1) / 2.5
        angles = np.arctan2(relative_vectors[:, 1], relative_vectors[:, 0]) / np.pi

        formation_input = np.stack([distances, angles], axis=1)[:9]

        # Convert to Tensors
        target_pos_tensor = torch.tensor([target_pos_val], dtype=torch.float32).to(
            device
        )
        formation_tensor = torch.tensor([formation_input], dtype=torch.float32).to(
            device
        )

        with torch.no_grad():
            # Encode the "Job Requirement"
            f_embed = model.formation_tower(formation_tensor, target_pos_tensor)
            f_norm = f_embed / (torch.linalg.norm(f_embed, dim=1, keepdim=True) + 1e-8)

            # Score all players in the squad for THIS specific position
            scores = []
            for _, player_row in squad_df.iterrows():
                p_stats = torch.tensor(
                    [player_row[player_stats_cols].values], dtype=torch.float32
                ).to(device)
                p_embed = model.player_tower(p_stats)
                p_norm = p_embed / (
                    torch.linalg.norm(p_embed, dim=1, keepdim=True) + 1e-8
                )

                score = torch.matmul(f_norm, p_norm.T).item()
                scores.append((player_row["player_name"], score))

        # Sort and take top 5 for the hover tooltip
        scores.sort(key=lambda x: x[1], reverse=True)
        top_list = "<br>".join([f"{name}: {s:.3f}" for name, s in scores[:5]])
        all_rankings.append(top_list)

    # 3. Create Plotly Visualization
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=coords_np[:, 0],
            y=coords_np[:, 1],
            mode="markers+text",
            marker=dict(
                size=25, color="RoyalBlue", line=dict(width=2, color="DarkSlateGrey")
            ),
            text=[f"Pos {i + 1}" for i in range(10)],
            textposition="top center",
            hoverinfo="text",
            hovertext=[f"<b>Top Candidates:</b><br>{rank}" for rank in all_rankings],
        )
    )

    # Clean up the layout to look like a pitch
    fig.update_layout(
        title="Tactical Fit Analysis: Hover to see ideal players",
        xaxis=dict(title="Pitch Width (X)", range=[0, 10], showgrid=False),
        yaxis=dict(title="Pitch Length (Y)", range=[0, 14], showgrid=False),
        width=700,
        height=900,
        template="plotly_white",
    )

    fig.show()


# --- EXAMPLE USAGE ---
# Define a 4-2-3-1 (approximate coords)
modern_4231 = [
    (2, 3),
    (8, 3),
    (4, 3),
    (6, 3),  # Back 4
    (4, 6),
    (6, 6),  # Midfield 2
    (2, 9),
    (5, 9),
    (8, 9),  # Attacking Mid 3
    (5, 12),  # Striker
]

classic_442 = [
    # Back 4 (Defense)
    (2, 3),  # Left Back (LB)
    (4, 3),  # Left Center Back (LCB)
    (6, 3),  # Right Center Back (RCB)
    (8, 3),  # Right Back (RB)
    # Midfield 4
    (2, 7),  # Left Midfield (LM / LW)
    (4, 7),  # Left Central Midfield (LCM)
    (6, 7),  # Right Central Midfield (RCM)
    (8, 7),  # Right Midfield (RM / RW)
    # Front 2 (Strikers)
    (4, 11),  # Left Striker (LS)
    (6, 11),  # Right Striker (RS)
]

attacking_433 = [
    # Back 4
    (2, 3),  # Left Back (LB)
    (4, 3),  # Left Center Back (LCB)
    (6, 3),  # Right Center Back (RCB)
    (8, 3),  # Right Back (RB)
    # Midfield 3
    (3, 7),  # Defensive Midfielder (CDM) - The "Pivot"
    (5, 7),  # Left Central Midfielder (LCM)
    (7, 7),  # Right Central Midfielder (RCM)
    # Front 3
    (2, 11),  # Left Winger (LW)
    (8, 11),  # Right Winger (RW)
    (5, 11),  # Center Forward (ST)
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the Simple Model
model = TwoTower().to(device)
model_path = "best_soccer_model.pth"
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

visualize_tactical_fit(
    modern_4231, df_match_fc_bayern_munich, model, device, player_stats_cols
)
# visualize_tactical_fit(
#     attacking_433, df_match_fc_bayern_munich, model, device, player_stats_cols
# )

# Model 2

In [11]:
class PositionTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(31),
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class SimpleTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.position_tower = PositionTower()
        self.player_tower = PlayerTower()

    def forward(self, position, player):
        pos_emb = self.position_tower(position)
        plr_emb = self.player_tower(player)

        p_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)
        u_norm = plr_emb / (torch.linalg.norm(plr_emb, dim=1, keepdim=True) + 1e-8)

        return p_norm, u_norm

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the Simple Model
model = SimpleTwoTower().to(device)
model_path = "simple_soccer_model.pth"  # Ensure you saved the simple model here
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [17]:
import plotly.graph_objects as go
import numpy as np
import torch
import pandas as pd


def visualize_simple_tactical_fit(
    custom_coords, squad_df, model, device, player_stats_cols
):
    """
    custom_coords: List of tuples [(x, y), ...] for any number of positions
    squad_df: DataFrame containing the squad to rank
    """
    model.eval()

    # 1. Prepare Coordinates
    coords_np = np.array(custom_coords, dtype=float)

    # Normalize globally exactly like the simple model training
    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    all_rankings = []

    # 2. Process each position independently (Simple Model Logic)
    for i in range(len(coords_norm)):
        # Convert single position to tensor (Batch size 1)
        target_pos_tensor = torch.tensor([coords_norm[i]], dtype=torch.float32).to(
            device
        )

        with torch.no_grad():
            # Encode the 'Position Requirement'
            p_emb = model.position_tower(target_pos_tensor)
            p_norm = p_emb / (torch.linalg.norm(p_emb, dim=1, keepdim=True) + 1e-8)

            # Score all players in the squad
            scores = []
            for _, player_row in squad_df.iterrows():
                # Extract player stats
                p_stats = torch.tensor(
                    [player_row[player_stats_cols].values], dtype=torch.float32
                ).to(device)

                # Encode Player (BatchNorm here uses moving averages due to model.eval())
                u_emb = model.player_tower(p_stats)
                u_norm = u_emb / (torch.linalg.norm(u_emb, dim=1, keepdim=True) + 1e-8)

                # Calculate Similarity
                score = torch.matmul(p_norm, u_norm.T).item()
                scores.append((player_row["player_name"], score))

        # Sort leaderboard for this specific dot
        scores.sort(key=lambda x: x[1], reverse=True)
        top_list = "<br>".join([f"{name}: {s:.4f}" for name, s in scores[:5]])
        all_rankings.append(top_list)

    # 3. Create Plotly Visualization
    fig = go.Figure()

    # Draw the pitch boundaries for context
    fig.add_shape(type="rect", x0=0, y0=0, x1=10, y1=14, line=dict(color="Black"))

    fig.add_trace(
        go.Scatter(
            x=coords_np[:, 0],
            y=coords_np[:, 1],
            mode="markers+text",
            marker=dict(
                size=30,
                color="FireBrick",  # Changed color to distinguish from advanced model
                line=dict(width=2, color="DarkSlateGrey"),
                opacity=0.8,
            ),
            text=[f"Pos {i + 1}" for i in range(len(custom_coords))],
            textposition="top center",
            hoverinfo="text",
            hovertext=[
                f"<b>Top Fits (Simple Model):</b><br>{rank}" for rank in all_rankings
            ],
        )
    )

    fig.update_layout(
        title="Simple Model: Tactical Fit by Coordinate (No Context)",
        xaxis=dict(title="Width", range=[-1, 11], showgrid=False, zeroline=False),
        yaxis=dict(title="Length", range=[-1, 15], showgrid=False, zeroline=False),
        width=700,
        height=900,
        template="plotly_white",
    )

    fig.show()


# Ensure your model is the SimpleTwoTower and loaded with 'simple_soccer_model.pth'
visualize_simple_tactical_fit(
    attacking_433, df_match_real_madrid, model, device, player_stats_cols
)

In [ ]:
def build_player_position_score_matrix(
    custom_coords,
    squad_df,
    model,
    device,
    player_stats_cols,
):
    """
    Returns a DataFrame of shape (num_players, num_positions)

    Rows: player names
    Columns: pos_1 ... pos_k
    Values: cosine similarity scores between player and position embeddings
    """

    model.eval()

    # ---------------------------
    # 1. Normalize coordinates
    # ---------------------------
    coords_np = np.array(custom_coords, dtype=float)

    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    pos_tensor = torch.tensor(coords_norm, dtype=torch.float32).to(device)

    # ---------------------------
    # 2. Encode all positions
    # ---------------------------
    with torch.no_grad():
        pos_emb = model.position_tower(pos_tensor)
        pos_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)

    # ---------------------------
    # 3. Encode all players
    # ---------------------------
    player_stats = squad_df[player_stats_cols].values
    player_tensor = torch.tensor(player_stats, dtype=torch.float32).to(device)

    with torch.no_grad():
        player_emb = model.player_tower(player_tensor)
        player_norm = player_emb / (
            torch.linalg.norm(player_emb, dim=1, keepdim=True) + 1e-8
        )

    # ---------------------------
    # 4. Similarity Matrix
    # ---------------------------
    # (num_players × embedding) @ (embedding × num_positions)
    sim_matrix = torch.matmul(player_norm, pos_norm.T).cpu().numpy()

    # ---------------------------
    # 5. Create DataFrame
    # ---------------------------
    pos_cols = [f"pos_{i + 1}" for i in range(len(custom_coords))]

    df_scores = pd.DataFrame(
        sim_matrix,
        index=squad_df["player_name"].values,
        columns=pos_cols,
    )

    return df_scores

In [26]:
df_scores_fc_bayern_4231 = build_player_position_score_matrix(
    modern_4231,
    df_match_fc_bayern_munich,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fc_bayern_4231)
df_scores_fc_bayern_4231.to_csv("fc_bayern_4231.csv")

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Arjen Robben        0.193862  0.367793  0.103793  0.228570  0.214119   
Philipp Lahm        0.340591  0.611393  0.410900  0.514872  0.428427   
Arturo Vidal        0.278365  0.545900  0.344865  0.450452  0.416020   
Robert Lewandowski  0.135318  0.284621  0.117983  0.235096  0.125763   
Thomas Mueller      0.291326  0.428070  0.193887  0.303041  0.252048   
David Alaba         0.352557  0.606026  0.340190  0.436209  0.464623   
Douglas Costa       0.251312  0.443619  0.150750  0.246110  0.310454   
Thiago Alcantara    0.196632  0.382895  0.220215  0.297341  0.402369   
Juan Bernat         0.352118  0.615397  0.290639  0.397649  0.400040   
Joshua Kimmich      0.191678  0.465166  0.260667  0.374594  0.570552   

                       pos_6     pos_7     pos_8     pos_9    pos_10  
Arjen Robben        0.307883  0.575293  0.556972  0.602692  0.404209  
Philipp Lahm        0.482616  0.319335  0.498631  0.437280  0.193

In [27]:
df_scores_fc_bayern_433 = build_player_position_score_matrix(
    attacking_433,
    df_match_fc_bayern_munich,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fc_bayern_433)
df_scores_fc_bayern_433.to_csv("fc_bayern_433.csv")

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Arjen Robben        0.193862  0.103793  0.228570  0.367793  0.383618   
Philipp Lahm        0.340591  0.410900  0.514872  0.611393  0.545302   
Arturo Vidal        0.278365  0.344865  0.450452  0.545900  0.477334   
Robert Lewandowski  0.135318  0.117983  0.235096  0.284621  0.363657   
Thomas Mueller      0.291326  0.193887  0.303041  0.428070  0.458371   
David Alaba         0.352557  0.340190  0.436209  0.606026  0.499371   
Douglas Costa       0.251312  0.150750  0.246110  0.443619  0.434889   
Thiago Alcantara    0.196632  0.220215  0.297341  0.382895  0.521536   
Juan Bernat         0.352118  0.290639  0.397649  0.615397  0.461514   
Joshua Kimmich      0.191678  0.260667  0.374594  0.465166  0.542885   

                       pos_6     pos_7     pos_8     pos_9    pos_10  
Arjen Robben        0.506284  0.518994  0.480474  0.490725  0.519741  
Philipp Lahm        0.675334  0.642158  0.272009  0.366344  0.307

In [28]:
df_scores_real_madrid_4231 = build_player_position_score_matrix(
    modern_4231,
    df_match_real_madrid,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fc_bayern_4231)
df_scores_fc_bayern_4231.to_csv("real_madrid_4231.csv")

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Arjen Robben        0.193862  0.367793  0.103793  0.228570  0.214119   
Philipp Lahm        0.340591  0.611393  0.410900  0.514872  0.428427   
Arturo Vidal        0.278365  0.545900  0.344865  0.450452  0.416020   
Robert Lewandowski  0.135318  0.284621  0.117983  0.235096  0.125763   
Thomas Mueller      0.291326  0.428070  0.193887  0.303041  0.252048   
David Alaba         0.352557  0.606026  0.340190  0.436209  0.464623   
Douglas Costa       0.251312  0.443619  0.150750  0.246110  0.310454   
Thiago Alcantara    0.196632  0.382895  0.220215  0.297341  0.402369   
Juan Bernat         0.352118  0.615397  0.290639  0.397649  0.400040   
Joshua Kimmich      0.191678  0.465166  0.260667  0.374594  0.570552   

                       pos_6     pos_7     pos_8     pos_9    pos_10  
Arjen Robben        0.307883  0.575293  0.556972  0.602692  0.404209  
Philipp Lahm        0.482616  0.319335  0.498631  0.437280  0.193

In [29]:
df_scores_fc_bayern_433 = build_player_position_score_matrix(
    attacking_433,
    df_match_real_madrid,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fc_bayern_433)
df_scores_fc_bayern_433.to_csv("real_madrid_433.csv")

                      pos_1     pos_2     pos_3     pos_4     pos_5     pos_6  \
Pepe               0.342455  0.513097  0.681713  0.619714  0.446762  0.622146   
Karim Benzema      0.148596  0.047739  0.183083  0.294706  0.327116  0.466517   
Marcelo            0.378177  0.340583  0.443844  0.603705  0.457762  0.636996   
Cristiano Ronaldo  0.203729  0.053722  0.186110  0.349725  0.342355  0.508293   
Sergio Ramos       0.374338  0.487165  0.600297  0.557940  0.397706  0.588813   
Luka Modric        0.228239  0.288896  0.379852  0.484565  0.541067  0.661355   
Gareth Bale        0.287598  0.127764  0.242118  0.465718  0.436628  0.594582   
Toni Kroos         0.198376  0.273636  0.331197  0.420630  0.505446  0.728135   
Casemiro           0.205977  0.358612  0.479839  0.459198  0.519684  0.723957   
Daniel Carvajal    0.488089  0.362581  0.492675  0.680391  0.463770  0.629878   

                      pos_7     pos_8     pos_9    pos_10  
Pepe               0.502090  0.135584  0.264095 